# High-Level Task APIs in K3-Node

K3-Node provides scikit-learn-style high-level Task estimators that allow you to train, evaluate, and predict on graphs in just 3 to 5 lines of code.

Supported Task APIs include:
- `NodeClassifier`: Single-label & multi-label node classification (`gcn`, `gat`, `sage`, `gin`, `pna`, `linkx`, etc.)
- `NodeRegressor`: Continuous node property prediction
- `GraphClassifier`: Whole-graph classification with readout pooling (`mean`, `add`, `max`)
- `GraphRegressor`: Molecular & graph-level continuous property regression (`schnet`, `dimenet`, `sage`, `gin`)
- `LinkPredictor`: Edge existence prediction with inner-product, cosine, or MLP decoders

The single cell below loads benchmark datasets, demonstrates each high-level task, and reports performance metrics on the Keras 3 multi-backend engine.

In [ ]:
# Setup environment and dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import ops
import numpy as np

import k3_node
from k3_node.datasets import Planetoid, TUDataset
from k3_node.tasks import (
    NodeClassifier,
    NodeRegressor,
    GraphClassifier,
    GraphRegressor,
    LinkPredictor,
)

backend = keras.config.backend()
print(f"[K3-Node] Running High-Level Tasks Showcase on Keras 3 ({backend}) backend...\n")

# ==============================================================================
# 1. Node Classification with GCN on Cora
# ==============================================================================
print("--- 1. Node Classification (Cora) ---")
cora_dataset = Planetoid(root="./data/Planetoid", name="Cora")
cora_data = cora_dataset[0]

clf = NodeClassifier(backbone="gcn", hidden_channels=64, num_layers=2, dropout=0.5)
clf.fit(cora_data, epochs=15, lr=0.01, verbose=0)
node_acc = clf.evaluate(cora_data, mask="test_mask")["accuracy"]
print(f"NodeClassifier (GCN) Test Accuracy: {node_acc:.4f}\n")

# ==============================================================================
# 2. Link Prediction with GCN on Cora
# ==============================================================================
print("--- 2. Link Prediction (Cora) ---")
lp = LinkPredictor(backbone="gcn", hidden_channels=64, out_channels=32, decoder="inner_product")
lp.fit(cora_data, epochs=10, lr=0.01, neg_ratio=1.0, verbose=0)
lp_res = lp.evaluate(cora_data)
acc_val = lp_res.get("accuracy", 0.0)
print(f"LinkPredictor Accuracy: {acc_val:.4f}", end="")
if "auc" in lp_res:
    auc_val = lp_res["auc"]
    ap_val = lp_res["ap"]
    print(f", ROC-AUC: {auc_val:.4f}, AP: {ap_val:.4f}")
else:
    print()
print()

# ==============================================================================
# 3. Graph Classification with GIN on MUTAG
# ==============================================================================
print("--- 3. Graph Classification (MUTAG) ---")
mutag_dataset = TUDataset(root="./data/MUTAG", name="MUTAG")
train_graphs = mutag_dataset[:140]
test_graphs = mutag_dataset[140:]

gc = GraphClassifier(backbone="gin", hidden_channels=64, num_layers=3, pooling="mean", dropout=0.5)
gc.fit(train_graphs, epochs=10, batch_size=32, lr=0.01, verbose=0)
gc_acc = gc.evaluate(test_graphs, batch_size=32)["accuracy"]
print(f"GraphClassifier (GIN) Test Accuracy: {gc_acc:.4f}\n")

# ==============================================================================
# 4. Graph Regression with GraphSAGE on Synthetic Molecular Graphs
# ==============================================================================
print("--- 4. Graph Regression (GraphSAGE) ---")
synth_graphs = []
for i in range(40):
    gx = np.random.randn(8, 16).astype("float32")
    gedge = np.array([[0, 1, 2, 3, 4, 5, 6, 7], [1, 2, 3, 4, 5, 6, 7, 0]], dtype="int64")
    gy = np.array([float(np.mean(gx) * 2.5)], dtype="float32")
    synth_graphs.append(k3_node.data.Data(x=gx, edge_index=gedge, y=gy))

gr = GraphRegressor(backbone="sage", hidden_channels=32, num_layers=2, pooling="mean")
gr.fit(synth_graphs[:30], epochs=10, batch_size=8, verbose=0)
gr_metrics = gr.evaluate(synth_graphs[30:], batch_size=8)
mae_val = gr_metrics["mae"]
mse_val = gr_metrics["mse"]
print(f"GraphRegressor (GraphSAGE) Test MAE: {mae_val:.4f}, MSE: {mse_val:.4f}\n")

print("✓ All high-level tasks executed successfully!")
